# Paper 2 · Langkah 3: melatih detektor pada tiga kondisi

Notebook ini menjawab bagian kedua RQ1: **berapa kinerja detektor yang tersisa setelah jalan pintas format dihilangkan?**

Detektor ResNet50 dilatih dengan 5-fold pada tiga kondisi data:
1. **asli**: file seperti apa adanya di benchmark (AI dikompresi JPEG ±75, karya manusia 95; Stable Diffusion 768 piksel).
2. **normal**: semua gambar 256 × 256, disimpan ulang JPEG kualitas 90.
3. **riwayat_disamakan**: karya manusia dikompresi dulu dengan kualitas 75 (sama seperti gambar AI), baru dinormalkan.

Ditambah uji lintas generator: latih pada satu generator, uji pada generator lain.

**Sebelum Run all:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Hasil tiap model langsung disimpan ke Drive. Kalau sesi terputus, klik Run all lagi; model yang sudah selesai dilewati.

## 1. Persiapan dan cek GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, io, json, time, shutil, zipfile, random, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torchvision
from PIL import Image
import PIL

# Pengaturan (jangan diubah tanpa mencatatnya di log kerja)
SEED = 42
MAX_EPOCHS = 30
PATIENCE = 5
BATCH = 32
LR = 1e-4
LR_MIN = 1e-6
INPUT = 224
PRETRAINED = True

PROJECT = Path('/content/drive/MyDrive/Paper2')
RAW = PROJECT / 'data_asli' / 'ai_artbench_sample'
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
RUNS = RESULTS / 'detektor_runs'
MODELS = RESULTS / 'models'
CACHE = PROJECT / 'data_olahan'
LOCAL = Path('/content/work')
for p in [RUNS, MODELS, CACHE, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'TIDAK ADA'
print('GPU:', gpu)
assert device == 'cuda', 'GPU belum aktif. Buka Runtime → Change runtime type → pilih T4 GPU → Save, lalu Run all lagi.'

env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\ntorch: {torch.__version__}\n'
       f'torchvision: {torchvision.__version__}\npillow: {PIL.__version__}\ngpu: {gpu}\n'
       f'seed: {SEED}, epoch maks: {MAX_EPOCHS}, patience: {PATIENCE}, batch: {BATCH}, lr: {LR}\n')
(NOTES / 'lingkungan_langkah3.txt').write_text(env)
print(env)

## 2. Salin data ke disk Colab

Kali pertama, 3.000 file disalin dari Drive (beberapa menit), lalu disimpan sebagai satu file zip di Drive. Kali berikutnya cukup membuka zip itu, jadi jauh lebih cepat.

In [ ]:
manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
assert len(manifest) == 3000 and 'fold' in manifest.columns, 'Jalankan notebook langkah 2 dulu.'
manifest['y'] = (manifest['label'] == 'ai').astype(int)

local_raw = LOCAL / 'raw'
raw_zip = CACHE / 'ai_artbench_sample_raw.zip'
t0 = time.time()
if not local_raw.exists():
    if raw_zip.exists():
        shutil.copy(raw_zip, LOCAL / 'raw.zip')
        with zipfile.ZipFile(LOCAL / 'raw.zip') as z:
            z.extractall(local_raw)
        print('Data dibuka dari zip di Drive.')
    else:
        files = [p for p in RAW.rglob('*') if p.is_file()]
        def _copy(p):
            d = local_raw / p.relative_to(RAW)
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(p, d)
        with ThreadPoolExecutor(16) as ex:
            list(ex.map(_copy, files))
        with zipfile.ZipFile(LOCAL / 'raw.zip', 'w', zipfile.ZIP_STORED) as z:
            for p in local_raw.rglob('*'):
                if p.is_file():
                    z.write(p, p.relative_to(local_raw))
        shutil.copy(LOCAL / 'raw.zip', raw_zip)
        print('Data disalin dari Drive dan disimpan sebagai zip untuk lain kali.')

manifest['raw_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)
missing = [p for p in manifest['raw_path'] if not os.path.exists(p)]
assert not missing, f'{len(missing)} file tidak ditemukan, contoh: {missing[:3]}'
print(f'3.000 file siap ({time.time() - t0:.0f} detik).')

## 3. Siapkan tiga kondisi data

Kondisi **normal** dibuat ulang dengan prosedur yang sama persis seperti langkah 2. Kondisi **riwayat_disamakan** menambah satu langkah untuk karya manusia: dikompresi dulu dengan JPEG kualitas 75 dan subsampling 4:2:0, seperti gambar AI di benchmark.

Semua gambar lalu diubah ke 224 × 224 dan dimuat ke memori supaya latihan cepat.

In [ ]:
def to_norm(im):
    im = im.convert('RGB')
    w, h = im.size
    if w != h:
        s = min(w, h); left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    buf = io.BytesIO()
    im.save(buf, 'JPEG', quality=90, subsampling=0, optimize=False)
    return buf.getvalue()

def make_condition(row, cond):
    with Image.open(row.raw_path) as im:
        im.load()
        if cond == 'asli':
            return im.convert('RGB')
        if cond == 'riwayat_disamakan' and row.source == 'human':
            buf = io.BytesIO()
            im.convert('RGB').save(buf, 'JPEG', quality=75, subsampling=2, optimize=False)
            im = Image.open(io.BytesIO(buf.getvalue())); im.load()
        return Image.open(io.BytesIO(to_norm(im))).convert('RGB')

def load_condition(cond):
    def _one(row):
        im = make_condition(row, cond)
        if im.size != (INPUT, INPUT):
            im = im.resize((INPUT, INPUT), Image.BICUBIC)
        return np.asarray(im, dtype=np.uint8)
    with ThreadPoolExecutor(8) as ex:
        arr = list(ex.map(_one, manifest.itertuples()))
    return torch.from_numpy(np.stack(arr)).permute(0, 3, 1, 2).contiguous()   # N,3,224,224 uint8

CONDITIONS = ['asli', 'normal', 'riwayat_disamakan']
DATA = {}
for c in CONDITIONS:
    t0 = time.time(); DATA[c] = load_condition(c)
    print(f'{c}: {tuple(DATA[c].shape)} ({time.time() - t0:.0f} detik)')

## 4. Fungsi latihan

ResNet50 pra-latih ImageNet, lapisan akhir diganti dua kelas, seluruh jaringan dilatih. Adam (1e-4), cosine annealing, cross-entropy, maksimal 30 epoch, berhenti lebih awal kalau loss validasi tidak membaik 5 epoch berturut-turut. Augmentasi hanya flip horizontal.

Di setiap fold, 10% dari data latih disisihkan untuk validasi. Fold uji hanya dipakai sekali, untuk evaluasi akhir.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
USE_AMP = device == 'cuda'
strata_all = (manifest['source'] + '|' + manifest['style']).values
y_all = manifest['y'].values

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def prep(xb, train):
    xb = xb.to(device, non_blocking=True).float().div_(255)
    if train:
        flip = torch.rand(xb.size(0), device=device) < 0.5
        xb[flip] = xb[flip].flip(-1)
    return ((xb - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def build_model():
    weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if PRETRAINED else None
    m = torchvision.models.resnet50(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(device).to(memory_format=torch.channels_last)

@torch.no_grad()
def predict(model, X, idx):
    model.eval(); out, losses = [], []
    for i in range(0, len(idx), 128):
        b = idx[i:i + 128]
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(prep(X[b], False))
        yb = torch.as_tensor(y_all[b], device=device)
        losses.append(nn.functional.cross_entropy(logits.float(), yb, reduction='sum').item())
        out.append(torch.softmax(logits.float(), 1)[:, 1].cpu().numpy())
    return np.concatenate(out), sum(losses) / len(idx)

def train_model(X, tr_idx, va_idx, seed):
    set_seed(seed)
    model = build_model()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=LR_MIN)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    rng = np.random.default_rng(seed)
    best, best_state, bad, hist = float('inf'), None, 0, []
    for ep in range(MAX_EPOCHS):
        model.train(); perm = rng.permutation(tr_idx)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            yb = torch.as_tensor(y_all[b], device=device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=USE_AMP):
                loss = nn.functional.cross_entropy(model(prep(X[b], True)), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        _, vloss = predict(model, X, va_idx)
        hist.append(round(vloss, 4))
        if vloss < best - 1e-4:
            best, bad = vloss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, hist

def metrics(y, p):
    pred = (p >= 0.5).astype(int)
    out = {'akurasi': accuracy_score(y, pred), 'presisi': precision_score(y, pred, zero_division=0),
           'recall': recall_score(y, pred, zero_division=0), 'f1': f1_score(y, pred, zero_division=0)}
    if len(np.unique(y)) == 2:
        out['roc_auc'] = roc_auc_score(y, p); out['pr_auc'] = average_precision_score(y, p)
    return {k: float(v) for k, v in out.items()}

def inner_split(tr_idx, seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=seed)
    try:      # seimbang menurut sumber dan gaya
        a, b = next(sss.split(tr_idx, strata_all[tr_idx]))
    except ValueError:   # data terlalu sedikit: seimbang menurut sumber saja
        a, b = next(sss.split(tr_idx, manifest['source'].values[tr_idx]))
    return tr_idx[a], tr_idx[b]

def run_and_save(name, X, tr_idx, te_idx, seed, save_weights):
    f = RUNS / f'{name}.json'
    if f.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    t0 = time.time()
    tr, va = inner_split(tr_idx, seed)
    model, hist = train_model(X, tr, va, seed)
    p, _ = predict(model, X, te_idx)
    rec = {'nama': name, 'epoch': len(hist), 'val_loss': hist, 'detik': round(time.time() - t0),
           'metrik': metrics(y_all[te_idx], p),
           'prediksi': [{'sample_id': manifest['sample_id'].iat[i], 'p_ai': round(float(pi), 6)} for i, pi in zip(te_idx, p)]}
    if save_weights:
        torch.save(model.state_dict(), MODELS / f'{name}.pt')
    f.write_text(json.dumps(rec))
    m = rec['metrik']
    print(f"{name}: akurasi {m['akurasi']:.3f}, AUC {m.get('roc_auc', float('nan')):.3f}, {rec['epoch']} epoch, {rec['detik']} detik")
    del model; torch.cuda.empty_cache()

## 5. Latihan 5-fold pada tiga kondisi (15 model)

Bobot model disimpan untuk kondisi **normal** dan **riwayat_disamakan**, karena dipakai di langkah peta atribusi. Bobot kondisi **asli** tidak disimpan untuk menghemat ruang Drive.

In [ ]:
folds = manifest['fold'].values
for cond in CONDITIONS:
    for k in range(5):
        tr_idx, te_idx = np.where(folds != k)[0], np.where(folds == k)[0]
        run_and_save(f'{cond}_fold{k}', DATA[cond], tr_idx, te_idx, SEED + k, save_weights=(cond != 'asli'))

## 6. Uji lintas generator (4 model)

Model dilatih pada karya manusia dan satu generator, lalu diuji pada generator lain. Karya manusia fold 0 sampai 3 dipakai untuk latih, fold 4 untuk uji, jadi tidak ada gambar uji yang pernah dilihat saat latihan.

In [ ]:
src = manifest['source'].values
human_tr, human_te = np.where((src == 'human') & (folds != 4))[0], np.where((src == 'human') & (folds == 4))[0]
PAIRS = [('latent_diffusion', 'stable_diffusion'), ('stable_diffusion', 'latent_diffusion')]
for cond in ['normal', 'riwayat_disamakan']:
    for train_gen, test_gen in PAIRS:
        tr_idx = np.concatenate([human_tr, np.where(src == train_gen)[0]])
        te_idx = np.concatenate([human_te, np.where(src == test_gen)[0]])
        short = {'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
        run_and_save(f'lintas_{cond}_latih{short[train_gen]}_uji{short[test_gen]}', DATA[cond], tr_idx, te_idx, SEED + 10, save_weights=False)

## 7. Ringkasan dan simpan hasil

In [ ]:
idx_of = {s: i for i, s in enumerate(manifest['sample_id'])}
rows, preds = [], []
for f in sorted(RUNS.glob('*.json')):
    r = json.loads(f.read_text())
    name = r['nama']
    ids = [idx_of[p['sample_id']] for p in r['prediksi']]
    p = np.array([q['p_ai'] for q in r['prediksi']])
    sub = manifest.iloc[ids]
    row = {'nama': name, 'epoch': r['epoch'], 'detik': r['detik'], **r['metrik']}
    # Per sumber: berapa persen tiap sumber ditebak benar
    for s in ['human', 'latent_diffusion', 'stable_diffusion']:
        mask = (sub['source'] == s).values
        if mask.any():
            row[f'benar_{s}'] = float(((p[mask] >= 0.5).astype(int) == sub['y'].values[mask]).mean())
    # AUC manusia vs tiap generator
    for s, tag in [('latent_diffusion', 'auc_manusia_vs_LD'), ('stable_diffusion', 'auc_manusia_vs_SD')]:
        mask = sub['source'].isin(['human', s]).values
        if mask.any() and len(np.unique(sub['y'].values[mask])) == 2:
            row[tag] = float(roc_auc_score(sub['y'].values[mask], p[mask]))
    rows.append(row)
    for i, pi in zip(ids, p):
        preds.append({'nama': name, 'sample_id': manifest['sample_id'].iat[i], 'label': manifest['label'].iat[i],
                      'source': manifest['source'].iat[i], 'style': manifest['style'].iat[i],
                      'fold': int(manifest['fold'].iat[i]), 'p_ai': pi, 'pred': 'ai' if pi >= 0.5 else 'human'})

runs = pd.DataFrame(rows)
runs.to_csv(RESULTS / 'rq1_detektor_per_model.csv', index=False)
pd.DataFrame(preds).to_csv(RESULTS / 'rq1_detektor_prediksi.csv', index=False)

cv = runs[~runs['nama'].str.startswith('lintas')].copy()
cv['kondisi'] = cv['nama'].str.replace(r'_fold\d$', '', regex=True)
cols = ['akurasi', 'presisi', 'recall', 'f1', 'roc_auc', 'pr_auc', 'benar_human', 'benar_latent_diffusion',
        'benar_stable_diffusion', 'auc_manusia_vs_LD', 'auc_manusia_vs_SD']
summary = cv.groupby('kondisi')[cols].agg(['mean', 'std']).round(4).reindex(CONDITIONS)
summary.to_csv(RESULTS / 'rq1_detektor_ringkasan.csv')

print('Jumlah model selesai:', len(runs), 'dari 19')
print('\nRINGKASAN 5-FOLD (rata-rata, dalam kurung simpangan baku)')
show = pd.DataFrame({c: summary[c]['mean'].map('{:.3f}'.format) + ' (' + summary[c]['std'].map('{:.3f}'.format) + ')' for c in cols})
display(show.T)
print('\nUJI LINTAS GENERATOR')
display(runs[runs['nama'].str.startswith('lintas')][['nama', 'akurasi', 'roc_auc', 'benar_human', 'benar_latent_diffusion', 'benar_stable_diffusion']].round(3))
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tertulis "Jumlah model selesai: 19 dari 19", semua beres. Tekan **Cmd+S**, lalu kabari Claude.

Kalau jumlahnya kurang dari 19 (misalnya sesi terputus atau jatah GPU habis), klik **Run all** lagi nanti. Model yang sudah selesai akan dilewati.